# Ticket Health

This notebook reads cached ticket data from DuckDB and frames a balanced scorecard across backlog, service quality, SLA health, hygiene, and ownership concentration.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

import plotly.express as px

from dynamix_manager.metrics import summarize_ticket_health

db_path = Path('/Users/micahcooper/dynamix-manager/data/analytics.duckdb')
connection = duckdb.connect(str(db_path), read_only=True)


In [2]:
tickets = connection.execute(
    "select * from tickets"
).fetchdf()
days_off_exists = connection.execute(
    "select count(*) from information_schema.tables where table_schema = 'main' and table_name = 'days_off'"
).fetchone()[0] > 0
days_off = connection.execute("select * from days_off").fetchdf() if days_off_exists else pd.DataFrame(columns=['holiday_date'])
quality_flags_exists = connection.execute(
    "select count(*) from information_schema.tables where table_schema = 'main' and table_name = 'ticket_quality_flags'"
).fetchone()[0] > 0
ticket_quality_flags = connection.execute("select * from ticket_quality_flags").fetchdf() if quality_flags_exists else pd.DataFrame()
quality_interactions_exists = connection.execute(
    "select count(*) from information_schema.tables where table_schema = 'main' and table_name = 'ticket_quality_interactions'"
).fetchone()[0] > 0
ticket_quality_interactions = connection.execute("select * from ticket_quality_interactions").fetchdf() if quality_interactions_exists else pd.DataFrame()
if 'holiday_date' in days_off.columns:
    days_off['holiday_date'] = pd.to_datetime(days_off['holiday_date'], utc=True, errors='coerce').dt.strftime('%Y-%m-%d')
business_holidays = set(days_off['holiday_date'].dropna()) if 'holiday_date' in days_off.columns else set()
for column in ['created_at', 'responded_at', 'resolved_at']:
    if column in tickets.columns:
        tickets[column] = pd.to_datetime(tickets[column], utc=True, errors='coerce')
for column in ['sla_begin_at', 'respond_by_at', 'resolve_by_at']:
    if column in tickets.columns:
        tickets[column] = pd.to_datetime(tickets[column], utc=True, errors='coerce')
if 'response_time_hours' not in tickets.columns and {'created_at', 'responded_at'}.issubset(tickets.columns):
    tickets['response_time_hours'] = (tickets['responded_at'] - tickets['created_at']).dt.total_seconds() / 3600
if 'resolution_time_hours' not in tickets.columns and {'created_at', 'resolved_at'}.issubset(tickets.columns):
    tickets['resolution_time_hours'] = (tickets['resolved_at'] - tickets['created_at']).dt.total_seconds() / 3600
if 'last_public_interaction_at' in ticket_quality_flags.columns:
    ticket_quality_flags['last_public_interaction_at'] = pd.to_datetime(ticket_quality_flags['last_public_interaction_at'], utc=True, errors='coerce')
if 'last_private_interaction_at' in ticket_quality_flags.columns:
    ticket_quality_flags['last_private_interaction_at'] = pd.to_datetime(ticket_quality_flags['last_private_interaction_at'], utc=True, errors='coerce')
if 'created_at' in ticket_quality_interactions.columns:
    ticket_quality_interactions['created_at'] = pd.to_datetime(ticket_quality_interactions['created_at'], utc=True, errors='coerce')
for column, default in {
    'client_last_interaction_flag': False,
    'it_follow_up_without_client_response_flag': False,
    'stale_public_update_flag': False,
    'private_activity_since_last_public_flag': False,
    'stale_public_update_business_days': 0,
    'last_private_interaction_at': pd.NaT,
    'last_private_interaction_by': pd.NA,
}.items():
    if column not in ticket_quality_flags.columns:
        ticket_quality_flags[column] = default
ticket_health_summary = summarize_ticket_health(tickets, days_off=days_off, quality_flags=ticket_quality_flags, interactions=ticket_quality_interactions)
tickets.head()

,ticket_id,ticket_title,status_name,status_class,type_name,priority_name,service_name,team_name,assignee_name,assignee_uid,...,sla_begin_at,respond_by_at,resolve_by_at,is_sla_violated,is_sla_respond_by_violated,is_sla_resolve_by_violated,ticket_app_id,ticket_app_name,response_time_hours,resolution_time_hours
0,5101130,Dawn Schluetz Office Move,Closed,3.0,Office Devices,Low,Office Devices: Request an Office Move,Tech Services,<NA>,7e7c4662-7fc5-e711-a95b-000d3a137856,...,2018-03-05 14:04:22.133000+00:00,NaT,2018-03-08 14:04:00+00:00,False,False,False,634,InfoTech Tickets,4.173696,50.478238
1,5103123,Dell Laptop in HR Conference Room not connecti...,Closed,3.0,Office Devices,Low,Office Devices: Ask for Help / Report a Problem,User Services,<NA>,bae4114f-c4aa-e711-a953-000d3a1afc30,...,NaT,NaT,2018-03-08 15:48:00+00:00,False,False,False,634,InfoTech Tickets,NaN,NaN
2,5103662,AKD Honor Society Induction,Closed,3.0,Classroom & Labs,Low,Classrooms & Labs: Request a Setup,Tech Services,<NA>,6c5ba3a1-7ec5-e711-a95b-000d3a137856,...,2018-03-05 16:17:06.117000+00:00,NaT,2018-03-19 19:05:00+00:00,True,False,True,634,InfoTech Tickets,313.875777,344.276325
3,5105493,Error message in VOUM of Colleague,Closed,3.0,Administrative Systems,Low,⚠️ Report an Issue with an Administrative System,Information Systems & Services (ISS),<NA>,fd2ad475-afaa-e711-a953-000d3a1afc30,...,2018-03-05 17:59:50.207000+00:00,NaT,2018-03-08 17:59:00+00:00,False,False,False,634,InfoTech Tickets,0.262512,3.627109
4,5107948,New employee's ID# may have been deleted from RE,Closed,3.0,Administrative Systems,Low,⚠️ Report an Issue with an Administrative System,Information Systems & Services (ISS),<NA>,f55545c0-ffac-e711-a953-000d3a1afc30,...,2018-03-05 20:13:10.227000+00:00,NaT,2018-03-08 20:13:00+00:00,False,False,False,634,InfoTech Tickets,17.375613,17.375613


## Quality Scorecard

In [3]:
pd.DataFrame([
    {'metric': 'Avg touches per ticket', 'value': ticket_health_summary['average_touches_per_ticket']},
    {'metric': 'Client last interaction', 'value': ticket_health_summary['quality_counts']['client_last_interaction']},
    {'metric': 'Repeated IT follow-up', 'value': ticket_health_summary['quality_counts']['repeated_it_followup']},
    {'metric': 'Stale Public Update', 'value': ticket_health_summary['quality_counts']['stale_public_updates']},
    {'metric': 'Private Activity Since Last Public Update', 'value': ticket_health_summary['quality_counts']['private_activity_since_last_public']},
    {'metric': 'Stale open tickets', 'value': ticket_health_summary['quality_counts']['stale_open_tickets']},
])

,metric,value
0,Avg touches per ticket,NaN
1,Client last interaction,0.0
2,Repeated IT follow-up,0.0
3,Stale Public Update,0.0
4,Private Activity Since Last Public Update,0.0
5,Stale open tickets,0.0


## SLA Health

In [4]:
pd.DataFrame([
    {'metric': 'Covered tickets', 'value': ticket_health_summary['sla_summary']['covered_tickets']},
    {'metric': 'SLA violations', 'value': ticket_health_summary['sla_summary']['violated_tickets']},
    {'metric': 'Respond breached', 'value': ticket_health_summary['sla_summary']['respond_breached']},
    {'metric': 'Resolve breached', 'value': ticket_health_summary['sla_summary']['resolve_breached']},
    {'metric': 'Near respond due', 'value': ticket_health_summary['sla_summary']['open_near_respond_due']},
    {'metric': 'Near resolve due', 'value': ticket_health_summary['sla_summary']['open_near_resolve_due']},
])

,metric,value
0,Covered tickets,5365
1,SLA violations,815
2,Respond breached,0
3,Resolve breached,178
4,Near respond due,0
5,Near resolve due,0


## Ticket Hygiene

In [5]:
pd.DataFrame([
    {'metric': 'Missing title', 'value': ticket_health_summary['hygiene_counts']['missing_title']},
    {'metric': 'Missing service', 'value': ticket_health_summary['hygiene_counts']['missing_service']},
    {'metric': 'Missing team', 'value': ticket_health_summary['hygiene_counts']['missing_team']},
    {'metric': 'Open unassigned', 'value': ticket_health_summary['hygiene_counts']['open_unassigned']},
    {'metric': 'Missing priority', 'value': ticket_health_summary['hygiene_counts']['missing_priority']},
])

,metric,value
0,Missing title,0
1,Missing service,0
2,Missing team,0
3,Open unassigned,23
4,Missing priority,0


## Status Mix

In [6]:
status_mix = (
    tickets.groupby('status_name', dropna=False)
    .size()
    .sort_values(ascending=False)
    .to_frame('tickets')
)
status_mix

,tickets
status_name,
Closed,5837
Cancelled,2


In [7]:
status_mix_chart = status_mix.reset_index(names='status_name')
px.bar(
    status_mix_chart,
    x='status_name',
    y='tickets',
    title='Ticket Status Mix',
)


## Team Breakdown

In [8]:
team_breakdown = (
    tickets.groupby('team_name', dropna=False)
    .agg(
        tickets=('ticket_id', 'count'),
        avg_response_hours=('response_time_hours', 'mean'),
        avg_resolution_hours=('resolution_time_hours', 'mean'),
    )
    .sort_values('tickets', ascending=False)
    .head(15)
)
team_breakdown

,tickets,avg_response_hours,avg_resolution_hours
team_name,,,
Tech Services,1935,115.733210,276.566986
User Services,1219,7.300686,26.505326
Information Systems & Services (ISS),922,91.204422,172.498637
TechStop,506,21.774100,90.048992
Network Infrastructure Services,478,30.873223,135.450676
Network Students,338,5.474408,51.496948
Tech Services Students,148,60.909787,250.686099
User Services Staff,146,33.987322,401.470387
Classroom & Lab,110,16.565551,36.793304


In [9]:
team_breakdown_chart = team_breakdown.reset_index().head(10)
px.bar(
    team_breakdown_chart,
    x='team_name',
    y='tickets',
    color='avg_resolution_hours',
    title='Top Teams by Ticket Volume',
)


## Team Completions Per Business Day

In [10]:
business_day_tickets = (
    tickets.loc[tickets['resolved_at'].notna()]
    .assign(resolved_date=tickets.loc[tickets['resolved_at'].notna(), 'resolved_at'].dt.strftime('%Y-%m-%d'))
)
business_day_tickets = business_day_tickets.loc[business_day_tickets['resolved_at'].dt.weekday < 5]
if business_holidays:
    business_day_tickets = business_day_tickets.loc[~business_day_tickets['resolved_date'].isin(business_holidays)]
team_completion_daily = (
    business_day_tickets
    .groupby(['resolved_date', 'team_name'], dropna=False)
    .size()
    .reset_index(name='completed_tickets')
    .sort_values(['resolved_date', 'completed_tickets', 'team_name'], ascending=[False, False, True])
)
team_completion_daily.head(30)

,resolved_date,team_name,completed_tickets
4009,2026-03-26,Tech Services Students,1
4010,2026-03-26,TechStop,1
4011,2026-03-26,User Services,1
4008,2026-03-25,Network Students,1
4007,2026-03-24,Tech Services,1
4006,2026-03-23,Tech Services,3
4005,2026-03-20,TechStop,1
4004,2026-03-19,TechStop,3
4002,2026-03-18,Tech Services,1
4003,2026-03-18,User Services,1


In [11]:
team_completion_chart = team_completion_daily.head(100)
px.line(
    team_completion_chart,
    x='resolved_date',
    y='completed_tickets',
    color='team_name',
    title='Team Completions Per Business Day',
)


## Member Completions Per Business Day

In [12]:
member_completion_daily = (
    business_day_tickets
    .assign(
        assignee_name=business_day_tickets['assignee_name'].astype('string').fillna('Unassigned').replace('', 'Unassigned'),
        team_name=business_day_tickets['team_name'].astype('string').fillna('Unassigned team').replace('', 'Unassigned team'),
    )
    .groupby(['resolved_date', 'team_name', 'assignee_name'], dropna=False)
    .size()
    .reset_index(name='completed_tickets')
    .sort_values(['resolved_date', 'team_name', 'completed_tickets', 'assignee_name'], ascending=[False, True, False, True])
)
member_completion_daily.head(40)

,resolved_date,team_name,assignee_name,completed_tickets
4009,2026-03-26,Tech Services Students,Unassigned,1
4010,2026-03-26,TechStop,Unassigned,1
4011,2026-03-26,User Services,Unassigned,1
4008,2026-03-25,Network Students,Unassigned,1
4007,2026-03-24,Tech Services,Unassigned,1
4006,2026-03-23,Tech Services,Unassigned,3
4005,2026-03-20,TechStop,Unassigned,1
4004,2026-03-19,TechStop,Unassigned,3
4002,2026-03-18,Tech Services,Unassigned,1
4003,2026-03-18,User Services,Unassigned,1


## Backlog Aging

In [13]:
pd.DataFrame(ticket_health_summary['backlog_age_buckets'])

,bucket,tickets
0,11+ business days,21


## Service Breakdown

In [14]:
service_breakdown = (
    tickets.groupby('service_name', dropna=False)
    .agg(
        tickets=('ticket_id', 'count'),
        avg_response_hours=('response_time_hours', 'mean'),
        avg_resolution_hours=('resolution_time_hours', 'mean'),
    )
    .sort_values('tickets', ascending=False)
    .head(20)
)
service_breakdown

,tickets,avg_response_hours,avg_resolution_hours
service_name,,,
Office Devices: Ask for Help / Report a Problem,1038,75.949115,185.801244
❓Request help with software,581,6.966788,100.944933
IT Internal: General Ticket Submission,505,33.732736,176.010557
Classrooms & Labs: Ask for Help / Report a Problem,466,38.087674,129.383773
Personal Devices: Ask for Help,423,16.632439,84.847865
None,385,19.929467,77.845040
📋 Register my Wi-Fi device,293,1.456642,48.186775
Classrooms & Labs: Request a Setup,281,355.239661,507.776455
Accounts & Access: Request a change in access rights,250,52.275260,97.979227


## High-Touch Tickets

In [15]:
pd.DataFrame(ticket_health_summary['high_touch_tickets'])

""


## Team Quality Hotspots

In [16]:
pd.DataFrame(ticket_health_summary['team_quality_hotspots'])

""


## Stale Public Update

In [17]:
ticket_quality_flags.loc[
    ticket_quality_flags['stale_public_update_flag'].fillna(False),
    ['ticket_id', 'ticket_title', 'team_name', 'service_name', 'stale_public_update_business_days', 'last_public_interaction_at', 'last_public_interaction_by'],
].sort_values(['stale_public_update_business_days', 'last_public_interaction_at'], ascending=[False, False]).head(50)

,ticket_id,ticket_title,team_name,service_name,stale_public_update_business_days,last_public_interaction_at,last_public_interaction_by


## Private Activity Since Last Public Update

In [18]:
ticket_quality_flags.loc[
    ticket_quality_flags['private_activity_since_last_public_flag'].fillna(False),
    ['ticket_id', 'ticket_title', 'team_name', 'service_name', 'last_private_interaction_at', 'last_private_interaction_by', 'last_public_interaction_by'],
].sort_values(['last_private_interaction_at', 'ticket_id'], ascending=[False, True]).head(50)

,ticket_id,ticket_title,team_name,service_name,last_private_interaction_at,last_private_interaction_by,last_public_interaction_by


## Quality-Adjusted SLA

In [19]:
pd.DataFrame([
    {'metric': 'Breached and high touch', 'value': ticket_health_summary['quality_adjusted_sla']['breached_and_high_touch']},
    {'metric': 'Breached and client waiting', 'value': ticket_health_summary['quality_adjusted_sla']['breached_and_client_waiting']},
    {'metric': 'Breached and repeated IT follow-up', 'value': ticket_health_summary['quality_adjusted_sla']['breached_and_repeated_it_followup']},
])

,metric,value
0,Breached and high touch,0
1,Breached and client waiting,0
2,Breached and repeated IT follow-up,0


## SLA Hotspots

In [20]:
pd.DataFrame(ticket_health_summary['sla_hotspots'])

,team_name,covered_tickets,violated_tickets,respond_breached,resolve_breached
0,Tech Services,701,392,False,79
1,Information Systems & Services (ISS),398,211,False,41
2,Network Infrastructure Services,179,88,False,19
3,Tech Services Students,65,44,False,7
4,TechStop,116,34,False,17
5,Network Students,120,17,False,1
6,Classroom & Lab,35,11,False,5
7,CIO,9,8,False,4
8,User Services,302,6,False,4
9,User Services Staff,15,3,False,1


## Backlog Load Hotspots

In [21]:
pd.DataFrame(ticket_health_summary['member_backlog_hotspots'])

,team_name,assignee_name,backlog_tickets,average_backlog_age
0,User Services,Unassigned,9,741.857143
1,Tech Services,Unassigned,4,673.500000
2,Information Systems & Services (ISS),Unassigned,3,1462.333333
3,Network Students,Unassigned,3,971.000000
4,TechStop,Unassigned,2,935.500000
5,Network Infrastructure Services,Unassigned,1,1483.000000
6,Classroom & Lab,Unassigned,1,797.000000


## Recurring Issue Candidates

In [22]:
pd.DataFrame(ticket_health_summary['top_recurrent_titles'])

,ticket_title,service_name,tickets
0,Manual Registration Request,📋 Register my Wi-Fi device,288
1,Printer Toner Request,𝍅 Printer Toner,99
2,Password Reset Request,Accounts & Access: Request a change in access ...,60
3,Phishing Email Received,Report Phishing Email,30
4,Request to Activate Residence Hall Ethernet Port,❓ Ask for Help/Report a Problem with Wi-Fi & I...,24
5,Merge Moodle Sections Request,Merge Canvas Sections,16
6,Request Office Move,Office Devices: Request an Office Move,15
7,VPN access requested,Virtual Private Network (VPN) Access,10
8,Direct Call to IT Staff,IT Internal: Transfer call,9
9,Print Quota Refund Request,💸 Request Print Quota Refund,8


## Hygiene Gaps

In [23]:
pd.DataFrame(ticket_health_summary['hygiene_tickets'])

,ticket_id,ticket_title,team_name,service_name,assignee_name,issues
0,5103123,Dell Laptop in HR Conference Room not connecti...,User Services,Office Devices: Ask for Help / Report a Problem,Unassigned,1
1,10196068,Unable to enter time worked,Information Systems & Services (ISS),❓Request help with software,Unassigned,1
2,11626664,Sample Invoice Billing for Spring - Need Meal ...,Information Systems & Services (ISS),⚠️ Report an Issue with an Administrative System,Unassigned,1
3,11742159,Request to access Requisitions,User Services,Accounts & Access: Request a change in access ...,Unassigned,1
4,12917469,Dual Factor Authentication,Network Infrastructure Services,None,Unassigned,1
5,15703649,An issue with vlab,User Services,❓Request help with software,Unassigned,1
6,17796757,we need our summer teams IC access to pioneer ...,Information Systems & Services (ISS),Accounts & Access: Request a change in access ...,Unassigned,1
7,18157739,Not printing on both sides,Tech Services,❓ Ask for help/Report a problem with printing,Unassigned,1
8,18376238,Request to Activate Residence Hall Ethernet Port,Network Students,❓ Ask for Help/Report a Problem with Wi-Fi & I...,Unassigned,1
9,18425505,Trouble with Eduroam,TechStop,Personal Devices: Ask for Help,Unassigned,1


## Backlog Review

In [24]:
tickets.loc[
    tickets['resolved_at'].isna(),
    ['ticket_id', 'status_name', 'team_name', 'service_name', 'assignee_name', 'created_at', 'response_time_hours'],
].sort_values('created_at', ascending=True).head(25)

,ticket_id,status_name,team_name,service_name,assignee_name,created_at,response_time_hours
1184,10196068,Closed,Information Systems & Services (ISS),❓Request help with software,<NA>,2019-08-12 18:04:32.040000+00:00,NaN
1518,11626664,Closed,Information Systems & Services (ISS),⚠️ Report an Issue with an Administrative System,<NA>,2019-12-05 12:04:23.313000+00:00,NaN
1499,11742159,Closed,User Services,Accounts & Access: Request a change in access ...,<NA>,2019-12-16 15:33:49.957000+00:00,NaN
1720,12917469,Closed,Network Infrastructure Services,None,<NA>,2020-03-23 01:37:14.013000+00:00,NaN
1935,15703649,Closed,User Services,❓Request help with software,<NA>,2020-10-24 15:28:19.230000+00:00,NaN
2373,17796757,Closed,Information Systems & Services (ISS),Accounts & Access: Request a change in access ...,<NA>,2021-05-18 16:32:03.880000+00:00,NaN
2485,18157739,Closed,Tech Services,❓ Ask for help/Report a problem with printing,<NA>,2021-07-16 15:56:01.370000+00:00,NaN
2580,18376238,Closed,Network Students,❓ Ask for Help/Report a Problem with Wi-Fi & I...,<NA>,2021-08-16 03:45:31.513000+00:00,NaN
2496,18425505,Cancelled,TechStop,Personal Devices: Ask for Help,<NA>,2021-08-19 17:30:28.503000+00:00,NaN
5415,18444128,Closed,Network Students,📋 Register my Wi-Fi device,<NA>,2021-08-21 02:51:59.890000+00:00,NaN


## Stale Open Tickets

In [25]:
pd.DataFrame(ticket_health_summary['stale_open_tickets'])

""


## Time To Serve

In [26]:
tickets[['response_time_hours', 'resolution_time_hours']].describe()

,response_time_hours,resolution_time_hours
count,1372.000000,5794.000000
mean,61.774636,165.427245
std,291.553967,612.216838
min,0.000000,0.000000
25%,0.113392,0.772646
50%,2.391813,20.392677
75%,24.305245,111.597100
max,5952.988000,13438.383409
